# **Lab 03:** Transfer Learning with PyTorch and Model Improvement

## Graded Individual Activity:

In this activity, we will carry out a series of tests to evaluate the performance of our image classification model. The objective is to correctly identify whether an image contains the face of a specific student or corresponds to a background. To do this, we have defined two main labels:

##### **Label 1:** `Student Name`  
##### **Label 2:** `Background` (images that do not contain the student’s face)

#### **Tests:**

The tests will be conducted using different types of input images to verify the model’s accuracy in scenarios not seen during training:

1. **Input image:** A photo of the student that is not included in the training folder (take a photo on the day of the presentation using your webcam).  
   **Required output label:** `Student Name`

2. **Input image:** A background image that is not included in the training folder (the student presenting before you will give you a search keyword; you may download the image from Google Images).  
   **Required output label:** `Background`

3. **Input image:** A photo of a different face (not the student’s face) that is not included in the training folders (the student presenting before you will give you the name of a celebrity who resembles you; you may download the image from Google Images).  
   **Required output label:** `Background`

##### **Hyperparameter Testing:**

Additionally, you may explore the impact of various hyperparameters on the model’s performance. Examples of hyperparameters to adjust include:

- Increasing the number of images in the dataset  
- Changing the loss function  
- Changing the optimizer  
- Changing the number of training epochs  
- Changing the batch size  

These tests and adjustments will allow you to optimize the model to achieve the best possible accuracy in image classification.

##### **Results Presentation**

Students will present their results in class using a `streamlit` app. The final grade will depend on the successful completion of the 3 tests.

In [19]:
# Write your code here

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [20]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [21]:
train_dataset = datasets.ImageFolder("dataset/train", transform=train_transforms)
val_dataset = datasets.ImageFolder("dataset/val", transform=val_test_transforms)
test_dataset = datasets.ImageFolder("dataset/test", transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(train_dataset.class_to_idx)

{'Luis': 0, 'background': 1}


In [22]:
import torch.nn as nn
from torchvision import models

def create_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    for param in model.parameters():
        param.requires_grad = False
    
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)
    
    return model

In [23]:
def train_and_validate(model, train_loader, val_loader, criterion, optimizer, device, epochs=5):
    model.to(device)
    best_val_acc = 0
    
    for epoch in range(epochs):
        model.train()
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # Validation
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_acc = correct / total
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
    
    return best_val_acc

In [24]:
import random
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

search_space = {
    "lr": [1e-2, 1e-3, 1e-4],
    "batch_size": [16, 32],
    "optimizer": ["adam", "sgd"],
    "weight_decay": [0, 1e-4]
}

n_trials = 6  # número de combinaciones aleatorias
results = []

for trial in range(n_trials):
    
    lr = random.choice(search_space["lr"])
    batch_size = random.choice(search_space["batch_size"])
    opt_name = random.choice(search_space["optimizer"])
    wd = random.choice(search_space["weight_decay"])
    
    print(f"\nTrial {trial+1}")
    print(f"lr={lr}, batch_size={batch_size}, optimizer={opt_name}, weight_decay={wd}")
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    model = create_model()
    
    if opt_name == "adam":
        optimizer = torch.optim.Adam(model.fc.parameters(), lr=lr, weight_decay=wd)
    else:
        optimizer = torch.optim.SGD(model.fc.parameters(), lr=lr, weight_decay=wd)
    
    val_acc = train_and_validate(model, train_loader, val_loader, criterion, optimizer, device)
    
    results.append({
        "lr": lr,
        "batch_size": batch_size,
        "optimizer": opt_name,
        "weight_decay": wd,
        "val_acc": val_acc
    })

print("\nResultados:")
for r in results:
    print(r)


Trial 1
lr=0.001, batch_size=32, optimizer=adam, weight_decay=0

Trial 2
lr=0.01, batch_size=16, optimizer=adam, weight_decay=0

Trial 3
lr=0.001, batch_size=32, optimizer=adam, weight_decay=0.0001

Trial 4
lr=0.001, batch_size=32, optimizer=adam, weight_decay=0

Trial 5
lr=0.001, batch_size=16, optimizer=sgd, weight_decay=0

Trial 6
lr=0.001, batch_size=32, optimizer=sgd, weight_decay=0.0001

Resultados:
{'lr': 0.001, 'batch_size': 32, 'optimizer': 'adam', 'weight_decay': 0, 'val_acc': 0.8775510204081632}
{'lr': 0.01, 'batch_size': 16, 'optimizer': 'adam', 'weight_decay': 0, 'val_acc': 0.9591836734693877}
{'lr': 0.001, 'batch_size': 32, 'optimizer': 'adam', 'weight_decay': 0.0001, 'val_acc': 0.9387755102040817}
{'lr': 0.001, 'batch_size': 32, 'optimizer': 'adam', 'weight_decay': 0, 'val_acc': 0.8775510204081632}
{'lr': 0.001, 'batch_size': 16, 'optimizer': 'sgd', 'weight_decay': 0, 'val_acc': 0.8571428571428571}
{'lr': 0.001, 'batch_size': 32, 'optimizer': 'sgd', 'weight_decay': 0.00

In [25]:
best_config = max(results, key=lambda x: x["val_acc"])
print("\nMejor configuración encontrada:")
print(best_config)


Mejor configuración encontrada:
{'lr': 0.01, 'batch_size': 16, 'optimizer': 'adam', 'weight_decay': 0, 'val_acc': 0.9591836734693877}


In [26]:
model = create_model()
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [27]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [28]:
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.01,
    weight_decay=0
)

criterion = nn.CrossEntropyLoss()

In [29]:
epochs = 15

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} - Val Acc: {val_acc:.4f}")

Epoch 1/15 - Val Acc: 0.5918
Epoch 2/15 - Val Acc: 0.7347
Epoch 3/15 - Val Acc: 0.9388
Epoch 4/15 - Val Acc: 0.9592
Epoch 5/15 - Val Acc: 0.9388
Epoch 6/15 - Val Acc: 0.9796
Epoch 7/15 - Val Acc: 1.0000
Epoch 8/15 - Val Acc: 0.9592
Epoch 9/15 - Val Acc: 0.9592
Epoch 10/15 - Val Acc: 0.9592
Epoch 11/15 - Val Acc: 0.9592
Epoch 12/15 - Val Acc: 0.9592
Epoch 13/15 - Val Acc: 0.9592
Epoch 14/15 - Val Acc: 0.9592
Epoch 15/15 - Val Acc: 0.9592


In [30]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", correct / total)

Test Accuracy: 1.0


In [31]:
torch.save(model.state_dict(), "resnet18_luis.pth")

### **Grading Rubric (10 points)**

The Streamlit app should be working correctly during presentation to get points.

| Criterion | Points |
|----------|--------|
| Test 1: Correct classification of student's new photo (`Student Name`) | 2.5 |
| Test 2: Correct classification of unseen background image (`Background`) | 2.5 |
| Test 3: Correct classification of a different face (celebrity look-alike) as `Background` | 5 |
| **Total** | **10** |

### **Scoring Rules**

- Each test is graded as **pass (full points)** or **fail (0 points)**.
- If the model predicts the required label correctly → full points for that test.
- If the prediction is incorrect → 0 points for that test.
- Test 3 has the highest weight because it evaluates the model’s ability to generalize and avoid false positives.

---

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>